# Control de Calidad — Base Egresados-Retirados
**Script:** `07_control_calidad_NJC.ipynb`  
**Semillero de Análisis Econométrico — UNAL FCE**

Verifica tres condiciones sobre la base consolidada de egresados y retirados:

| # | Check | Variable(s) |
|---|---|---|
| 1 | Unicidad de la clave | `ID_UNAL`, `periodo`, `COD_PLAN` |
| 2 | Calificaciones en rango [0, 5] | `PROM_GRADUADO_RETIRADO` |
| 3 | Conflicto graduado=1 y retirado=1 | `graduado`, `retirado` |

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

from config import DIR_DATOS

DIR_OUTPUT = DIR_DATOS / "DatosArmonizados" / "base_egresados_retirados"
LLAVE = ["ID_UNAL", "periodo", "COD_PLAN"]

---
## 0. Carga de datos

In [ ]:
archivos = sorted(DIR_OUTPUT.glob("BASE_EGRESADOS_RETIRADOS_*.csv"))

if not archivos:
    raise FileNotFoundError(f"No se encontraron archivos en {DIR_OUTPUT}. "
                            "Ejecutar primero 04_base_egresados_retirados.py.")

df = pd.concat(
    [pd.read_csv(f, sep=";", dtype=str) for f in archivos],
    ignore_index=True,
)

print(f"Archivos cargados : {len(archivos)}")
print(f"Total filas       : {len(df):,}")
print(f"Total columnas    : {df.shape[1]}")
print(f"Periodos únicos   : {df['periodo'].nunique()}")
display(df.head(3))

---
## Check 1 — Unicidad de la clave `(ID_UNAL, periodo, COD_PLAN)`

In [ ]:
dup_mask   = df.duplicated(subset=LLAVE, keep=False)
n_dup_filas = dup_mask.sum()
n_dup_llaves = df[dup_mask].groupby(LLAVE).ngroups if n_dup_filas else 0

print(f"Filas totales           : {len(df):,}")
print(f"Filas con llave repetida: {n_dup_filas:,}")
print(f"Llaves únicas con dupes : {n_dup_llaves:,}")

if n_dup_filas == 0:
    display(Markdown("### ✅ OK — La clave `(ID_UNAL, periodo, COD_PLAN)` es única en toda la base."))
else:
    display(Markdown(
        f"### ❌ ERROR — **{n_dup_llaves:,}** llaves duplicadas → **{n_dup_filas:,}** filas afectadas"
    ))
    display(Markdown("**Primeras 30 filas con llave duplicada:**"))
    display(
        df[dup_mask]
        .sort_values(LLAVE)
        .head(30)
        .reset_index(drop=True)
    )

---
## Check 2 — Calificaciones en rango [0, 5]: `PROM_GRADUADO_RETIRADO`

In [ ]:
COL_PROM = "PROM_GRADUADO_RETIRADO"

# Las notas están almacenadas con coma decimal (ej. "3,7"); se normaliza antes de convertir
serie   = pd.to_numeric(df[COL_PROM].str.replace(",", ".", regex=False), errors="coerce")
n_nulos = serie.isna().sum()
n_total = len(serie)
mask_fuera = (serie < 0) | (serie > 5)
n_fuera    = mask_fuera.sum()

print(f"Columna            : {COL_PROM}")
print(f"Total filas        : {n_total:,}")
print(f"Nulos / sin nota   : {n_nulos:,}  ({n_nulos/n_total*100:.1f}%)")
print(f"Con valor numérico : {n_total - n_nulos:,}")
print(f"Fuera de [0, 5]    : {n_fuera:,}")

if n_fuera > 0:
    print()
    print("Distribución de los valores fuera de rango:")
    display(serie[mask_fuera].describe().rename(COL_PROM).to_frame())

print()
print("Estadísticas de los valores numéricos válidos:")
display(serie[~mask_fuera].describe().rename(COL_PROM).to_frame())

In [ ]:
if n_fuera == 0:
    display(Markdown(
        f"### ✅ OK — Todos los {n_total - n_nulos:,} valores numéricos de `{COL_PROM}` "
        "están dentro del rango [0, 5]."
    ))
else:
    display(Markdown(
        f"### ❌ ERROR — **{n_fuera:,}** valores de `{COL_PROM}` están fuera del rango 0–5"
    ))
    display(Markdown("**Filas con valor fuera de rango:**"))
    display(
        df[mask_fuera][LLAVE + [COL_PROM, "graduado", "retirado"]]
        .reset_index(drop=True)
    )

---
## Check 3 — Estudiantes con `graduado=1` y `retirado=1` simultáneamente

In [ ]:
grad  = pd.to_numeric(df["graduado"],  errors="coerce")
retir = pd.to_numeric(df["retirado"],  errors="coerce")
mask_conflicto = (grad == 1) & (retir == 1)
n_conflicto    = mask_conflicto.sum()

print(f"Filas con graduado=1             : {(grad  == 1).sum():,}")
print(f"Filas con retirado=1             : {(retir == 1).sum():,}")
print(f"Filas con graduado=1 y retirado=1: {n_conflicto:,}")

In [ ]:
if n_conflicto == 0:
    display(Markdown(
        "### ✅ OK — Ningún estudiante tiene `graduado=1` y `retirado=1` a la vez."
    ))
else:
    display(Markdown(
        f"### ⚠️ WARN — **{n_conflicto:,}** estudiantes tienen `graduado=1` y `retirado=1` simultáneamente.  \n"
        "Esto ocurre cuando el mismo par `(ID_UNAL, periodo, COD_PLAN)` aparece en ambas bases."
    ))
    display(Markdown("**Detalle de los casos:**"))
    display(
        df[mask_conflicto]
        .reset_index(drop=True)
    )

---
## Resumen ejecutivo

---
## Check 4 — Valores vacíos por columna y periodo

In [ ]:
def resumen_nulos(g):
    n_total = len(g)
    n_nulos = g.isna().sum()
    pct     = (n_nulos / n_total * 100).round(1)
    return n_nulos.astype(str) + " (" + pct.astype(str) + "%)"

tabla_nulos = (
    df
    .groupby("periodo", dropna=False)
    .apply(resumen_nulos)
    .drop(columns=["periodo"], errors="ignore")
)

# Fila de totales generales
n_nulos_total = df.isna().sum()
pct_total     = (n_nulos_total / len(df) * 100).round(1)
fila_total    = n_nulos_total.astype(str) + " (" + pct_total.astype(str) + "%)"
fila_total.name = "TOTAL"

tabla_nulos = pd.concat([tabla_nulos, fila_total.to_frame().T])

# Solo columnas que tienen al menos un nulo
columnas_con_nulos = df.columns[df.isna().any()].tolist()
tabla_nulos = tabla_nulos[columnas_con_nulos]

display(Markdown(f"**{len(columnas_con_nulos)} columnas con al menos un valor vacío "
                 f"(de {df.shape[1]} totales)**"))
display(tabla_nulos)

In [ ]:
def estado(ok: bool, warn: bool = False) -> str:
    if ok:   return "✅ OK"
    if warn: return "⚠️ WARN"
    return "❌ ERROR"

resumen = pd.DataFrame([
    {
        "Check": "Unicidad clave (ID_UNAL, periodo, COD_PLAN)",
        "Estado": estado(n_dup_filas == 0),
        "Detalle": "OK" if n_dup_filas == 0
                   else f"{n_dup_llaves:,} llaves / {n_dup_filas:,} filas duplicadas",
    },
    {
        "Check": "PROM_GRADUADO_RETIRADO en [0, 5]",
        "Estado": estado(n_fuera == 0),
        "Detalle": "OK" if n_fuera == 0
                   else f"{n_fuera:,} valores fuera de rango",
    },
    {
        "Check": "Sin graduado=1 y retirado=1 simultáneo",
        "Estado": estado(ok=(n_conflicto == 0), warn=False) if n_conflicto == 0
                   else "⚠️ WARN",
        "Detalle": "OK" if n_conflicto == 0
                   else f"{n_conflicto:,} casos de conflicto",
    },
])

display(resumen.style.hide(axis="index"))